<div style="background-color: #fff8e1; border-left: 5px solid #ffc107; padding: 10px; margin-bottom: 10px;">
  <b>참고:</b> CloudWatch에서 이미 Transaction Search를 활성화했다면 이 단계를 건너뛸 수 있습니다.
</div>

# Amazon Bedrock AgentCore Observability: Transaction Search 활성화
이 Notebook에서는 Amazon Bedrock AgentCore의 관찰성을 향상하기 위해 [Transaction Search](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/CloudWatch-Transaction-Search.html)를 활성화하는 방법을 살펴봅니다. Transaction Search는 애플리케이션 트랜잭션의 span을 완전하게 파악할 수 있는 대화형 분석 환경입니다. span은 분산 trace의 기본 작업 단위이며 애플리케이션이나 시스템의 특정 동작 또는 작업을 나타냅니다. 각 span에는 트랜잭션의 특정 구간에 대한 세부 정보가 기록됩니다. 이러한 세부 정보에는 시작 및 종료 시간, 지속 시간, 그리고 고객 ID나 주문 ID 같은 비즈니스 속성을 포함할 수 있는 관련 metadata가 포함됩니다. span은 부모-자식 계층 구조로 구성됩니다. 이 계층 구조는 트랜잭션이 여러 구성 요소나 서비스를 거쳐 흐르는 과정을 매핑하여 하나의 완전한 trace를 형성합니다.

Transaction Search는 콘솔, API 또는 AWS CloudFormation을 사용하여 활성화할 수 있습니다. Transaction Search는 계정 전체에 적용되며, X-Ray를 통한 모든 span 수집을 [Amazon CloudWatch 요금](https://aws.amazon.com/cloudwatch/pricing/)에 따른 비용 효율적인 수집 모드로 전환합니다. 또한 기본적으로 수집된 span의 1%를 분석용 trace 요약으로 무료 인덱싱합니다. Transaction Search를 통해 수집된 모든 span의 전체 end-to-end trace를 이미 확인할 수 있으므로 일반적으로 이 정도면 충분합니다.

# 학습 내용
- AWS CloudFormation을 사용하여 Transaction Search를 활성화하는 방법

# 사전 요구 사항
- [문서](https://docs.aws.amazon.com/IAM/latest/UserGuide/id_roles_update-role-permissions.html)에 따라 CLI 명령 실행, CloudFormation stack 생성 및 Transaction Search 활성화에 필요한 적절한 권한을 'SageMaker Execution Role'에 부여합니다.

# 1. 설정 및 구성
CloudFormation을 사용하여 Transaction Search를 활성화하려면 다음 두 리소스를 생성해야 합니다.

- AWS::Logs::ResourcePolicy
- AWS::XRay::TransactionSearchConfig

프로젝트 폴더에 제공된 샘플 [CloudFormation template](transaction_search.yml)을 필요에 맞게 검토하고 수정하세요. 아래 코드를 실행하여 resource policy를 생성하고, X-Ray가 trace를 CloudWatch Logs로 전송할 수 있도록 Transaction Search를 활성화합니다.

In [ ]:
!aws cloudformation create-stack --stack-name transaction-search --template-body file://transaction_search.yml

Transaction Search를 활성화하는 데는 5~10분이 걸립니다. stack 생성 상태가 `CREATE_COMPLETE`가 될 때까지 기다리세요. 아래 코드를 사용하여 상태를 확인할 수 있습니다.

In [ ]:
!aws cloudformation describe-stacks --stack-name transaction-search --query 'Stacks[0].StackStatus' --output text


# 2. 구성 확인

AWS CloudFormation stack을 배포한 후 아래 코드를 사용하여 구성을 확인할 수 있습니다. 구성이 성공하면 Destination은 'CloudWatchLogs', Status는 'ACTIVE'로 반환됩니다.



In [ ]:
!aws xray get-trace-segment-destination


다음과 같이 CloudWatch 콘솔에서도 확인할 수 있습니다.

- Transaction Search 활성화 전:

<div style="text-align:left">
    <img src="images/transaction_search_disabled1.png" width="100%"/>
</div>

<div style="text-align:left">
    <img src="images/transaction_search_disabled2.png" width="100%"/>
</div>

- Transaction Search 활성화 후:

<div style="text-align:left">
    <img src="images/transaction_search_enabled1.png" width="100%"/>
</div>

<div style="text-align:left">
    <img src="images/transaction_search_enabled2.png" width="100%"/>
</div>



# 3. 마무리

Transaction Search를 활성화하면 Application Signals와 CloudWatch Logs의 기능을 포함한 여러 기능을 사용할 수 있습니다. X-Ray로 전송된 span은 aws/spans라는 log group에 수집됩니다. CloudWatch는 이러한 span을 사용하여 CloudWatch Application Signals에서 선별된 application performance monitoring(APM) 환경을 제공합니다. 이를 통해 span을 검색하고 분석할 수 있으며, 이상치 및 패턴 탐지 같은 CloudWatch Logs 기능도 사용할 수 있습니다. custom metric 추출도 가능합니다. CloudWatch Application Signals는 애플리케이션, 서비스 및 종속성을 애플리케이션 중심의 통합된 관점에서 보여 줍니다. 또한 애플리케이션 상태를 모니터링하고 문제의 우선순위를 분류하는 데 도움이 됩니다.

이 모듈에서는 CloudFormation을 사용하여 Transaction Search를 활성화하는 방법을 다뤘습니다. 필요에 따라 이 [문서](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html)를 참고하여 콘솔, API 또는 CLI로 Transaction Search를 활성화할 수도 있습니다.


